# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will:

- Load schema and metadata
- Examine record sets and their schema (fields and columns, referenced by `@id`)
- Load the data into DataFrames
- Process and visualize the data

### Dataset Source
The dataset is sourced using a Croissant schema URL hosted at Sen Science.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Dataset object
dataset = mlc.Dataset(croissant_url)

# Show metadata summary
md = dataset.metadata  # single object, not subscripted
print(f"Dataset: {md.name}\nDescription: {md.description}\nIdentifier: {md.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All elements are referenced by their `@id`.

This dataset may contain multiple record sets, each describing a table of data. We'll enumerate them with their fields.

In [ ]:
# List all record sets and their fields using @id
print("Available Record Sets (by @id):\n-------------------------")
for rs in md.record_sets:
    print(f"- [@id]: {rs.id}   [name]: {rs.name}")
    field_ids = [field.id for field in rs.fields]
    print("  Fields (by @id):")
    for field in rs.fields:
        col_ids = [col.id for col in getattr(field, 'columns', [])]
        print(f"    - [@id]: {field.id}   [name]: {field.name}, columns: {col_ids if col_ids else 'N/A'}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are by `@id`.

> **Tip:** Use the record set and field `@id`s you found above.

Let's extract all tabular record sets into DataFrames.

In [ ]:
# Collect record set @id's
record_set_ids = [rs.id for rs in md.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # yield records as list of dict
    rows = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(rows)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}.")
    print(f"Sample columns: {df.columns.tolist()[:10]}")
    print()

## 4. Exploratory Data Analysis (EDA)
Now, let's select a record set and perform some basic processing using its field `@id`s only.

In [ ]:
# Pick the first (main) record set by @id
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

# List all columns in this DataFrame for reference
print(f"Available columns in record set {main_record_set_id}:")
for col in main_df.columns:
    print(f"- {col}")

# Let's select a numeric field for demonstration (e.g. coverage, MW, or similar)
# We'll try to find common mass spec relevant columns by their @id or name
candidates = [col for col in main_df.columns if any(k in col.lower() for k in ['coverage','mw','weight','abundance','count','peptide'])]
print(f'Numeric candidate columns: {candidates}')

# Pick one, say the first
if candidates:
    numeric_field_id = candidates[0]
else:
    numeric_field_id = main_df.columns[0]
    print(f"Warning: Defaulting to first column as numeric: {numeric_field_id}")

# Make sure it's numeric
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = main_df[numeric_field_id].mean()  # set a threshold at mean for demo
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered rows with {numeric_field_id} > {threshold:.2f} (N={filtered_df.shape[0]} records):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a likely categorical field
group_candidates = [col for col in main_df.columns if any(k in col.lower() for k in ['accession','description','group','sample','experiment','id']) and col != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using names/`@id`s.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution of the numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    filtered_df[numeric_field_id].hist(bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped, show bar chart
    if 'group_field_id' in locals():
        # Plot up to 20 groups
        grouped.head(20).plot(kind='bar', figsize=(10,5))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all data elements by their `@id` as per the schema. Further steps could include deeper statistical analysis or integration with other proteomics resources.